In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import requests
import re, re as regex
import time
import pandas as pd
import random
import nltk
from datetime import datetime
import ast
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Used for exporting .csv file to the Drive
import shutil
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [ ]:
articles_all = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/MAP6114 - Machine Learning/MAP 6114 Project/FINAL_articles_raw.csv")

articles_raw = articles_all.drop(columns=['text', 'summary', 'meta'])

articles_raw['keywords'] = articles_raw['keywords'].apply(ast.literal_eval)

# Convert lists to tuples because tuples are hashable and fast to count
display(articles_raw['keywords'].apply(tuple).value_counts().head(20))

print(f'Prior to cleaning, the DF has {len(articles_raw)} articles')

names = ['msn', 'zerohedge', 'redirected']

for name in names:
  articles_raw = articles_raw[~articles_raw['keywords'].apply(lambda x: x == [name])]

articles_raw = articles_raw[~articles_raw['keywords'].apply(lambda x: x == [])]
articles_raw = articles_raw.reset_index(drop=True)

print(f'After cleaning, the DF has {len(articles_raw)} articles')

,count
keywords,
"(oil, tsx, energy, business, president, markets, natural, waters, canada, soon, prices, wean, russian, trillion, today, ways, reach)",288
"(msn,)",238
"(redirected,)",188
"(daughters, financial, largest, sanctions, russian, putins, placed, united, bans, russia, bank, ukraine, states, investments, minister)",128
"(iran, plans, iranian, sanctions, officials, shakeri, plot, memorandum, reveals, assassinates, left, obliterate, instructed, zero, trump)",109
"(regions, announces, president, sanctions, russias, putin, russian, announced, biden, ukraine, invasion, russia, significant)",85
(),84
"(oil, power, tsx, planetwarming, energy, business, looking, toronto, hamilton, natural, canada, undermining, prices, pollution, today, report, increased)",81
"(zerohedge,)",73


Prior to cleaning, the DF has 40645 articles
After cleaning, the DF has 40062 articles


In [ ]:
display(articles_raw)

,title,date,keywords,url,country
0,National Defense Authorization Act – Implicati...,2025-11-19 00:00:00,"[provisions, authorization, uyghur, president,...",https://www.jdsupra.com/legalnews/national-def...,NaN
1,Full diplomatic break? Decoding the latest Bid...,2021-12-30 05:00:00+00:00,"[impose, break, ushakov, bidenputin, latest, s...",https://www.washingtonexaminer.com/restoring-a...,NaN
2,Canadian Bank Settles OFAC Charges For Potenti...,2025-11-19 00:00:00,"[settles, sanctions, apparent, kingpin, violat...",https://www.mondaq.com/unitedstates/export-con...,NaN
3,RAB wants govt to resolve US sanctions,2025-11-19 00:00:00,"[resolve, authorities, sanctions, officials, r...",https://en.prothomalo.com/bangladesh/rab-wants...,NaN
4,"Travel News: Latest News, Breaking Stories, an...",2022-01-03 00:00:00,"[latest, unless, india, breaking, rate, losses...",https://skift.com/2022/01/03/airbnb-settles-cu...,Cuba
...,...,...,...,...,...
40057,Rise up for Gaza rally held in Huntsville,2025-10-05 00:00:00,"[gaza, organizer, israel, waff, world, hamas, ...",https://www.waff.com/2025/10/05/rise-up-gaza-r...,NaN
40058,"In search of FOI that barks and bites, by Taiw...",2025-10-05 11:23:54+00:00,"[foi, adisa, law, implementation, barks, taiwo...",https://theeagleonline.com.ng/in-search-of-foi...,NaN
40059,"Stop the Malicious Lies, Onanuga Tackles US Se...",2025-10-05 00:00:00,"[lies, christians, country, mass, senator, thi...",https://www.thisdaylive.com/2025/10/05/stop-th...,NaN
40060,Ukraine war briefing: China providing Russia w...,2025-10-05 00:00:00,"[ukrainian, providing, claims, intelligence, m...",https://www.theguardian.com/world/2025/oct/05/...,NaN


In [ ]:
articles_raw["date"] = pd.to_datetime(articles_raw["date"], format='mixed', utc=True)
articles_raw["date"] = articles_raw["date"].dt.tz_convert(None)

In [ ]:
display(articles_raw)

,title,date,keywords,url,country
0,National Defense Authorization Act – Implicati...,2025-11-19 00:00:00,"[provisions, authorization, uyghur, president,...",https://www.jdsupra.com/legalnews/national-def...,NaN
1,Full diplomatic break? Decoding the latest Bid...,2021-12-30 05:00:00,"[impose, break, ushakov, bidenputin, latest, s...",https://www.washingtonexaminer.com/restoring-a...,NaN
2,Canadian Bank Settles OFAC Charges For Potenti...,2025-11-19 00:00:00,"[settles, sanctions, apparent, kingpin, violat...",https://www.mondaq.com/unitedstates/export-con...,NaN
3,RAB wants govt to resolve US sanctions,2025-11-19 00:00:00,"[resolve, authorities, sanctions, officials, r...",https://en.prothomalo.com/bangladesh/rab-wants...,NaN
4,"Travel News: Latest News, Breaking Stories, an...",2022-01-03 00:00:00,"[latest, unless, india, breaking, rate, losses...",https://skift.com/2022/01/03/airbnb-settles-cu...,Cuba
...,...,...,...,...,...
40057,Rise up for Gaza rally held in Huntsville,2025-10-05 00:00:00,"[gaza, organizer, israel, waff, world, hamas, ...",https://www.waff.com/2025/10/05/rise-up-gaza-r...,NaN
40058,"In search of FOI that barks and bites, by Taiw...",2025-10-05 11:23:54,"[foi, adisa, law, implementation, barks, taiwo...",https://theeagleonline.com.ng/in-search-of-foi...,NaN
40059,"Stop the Malicious Lies, Onanuga Tackles US Se...",2025-10-05 00:00:00,"[lies, christians, country, mass, senator, thi...",https://www.thisdaylive.com/2025/10/05/stop-th...,NaN
40060,Ukraine war briefing: China providing Russia w...,2025-10-05 00:00:00,"[ukrainian, providing, claims, intelligence, m...",https://www.theguardian.com/world/2025/oct/05/...,NaN


In [ ]:
#For ease of analysis set week to Monday
articles_raw["week_start"] = articles_raw["date"].dt.to_period("W-MON").dt.start_time
display(articles_raw[["date", "week_start"]].head())

,date,week_start
0,2025-11-19 00:00:00,2025-11-18
1,2021-12-30 05:00:00,2021-12-28
2,2025-11-19 00:00:00,2025-11-18
3,2025-11-19 00:00:00,2025-11-18
4,2022-01-03 00:00:00,2021-12-28


In [ ]:
display(articles_raw['keywords'].apply(tuple).value_counts().head(1000))

,count
keywords,
"(oil, tsx, energy, business, president, markets, natural, waters, canada, soon, prices, wean, russian, trillion, today, ways, reach)",288
"(daughters, financial, largest, sanctions, russian, putins, placed, united, bans, russia, bank, ukraine, states, investments, minister)",128
"(iran, plans, iranian, sanctions, officials, shakeri, plot, memorandum, reveals, assassinates, left, obliterate, instructed, zero, trump)",109
"(regions, announces, president, sanctions, russias, putin, russian, announced, biden, ukraine, invasion, russia, significant)",85
"(oil, power, tsx, planetwarming, energy, business, looking, toronto, hamilton, natural, canada, undermining, prices, pollution, today, report, increased)",81
...,...
"(department, religious, india, 300, indian, leaders, letter, christian, govt, state, demand, persecution, international, action)",2
"(financial, x10, drones, sanctions, company, national, drone, war, ongoing, china, skydio, chinese, trade, maker)",2
"(democrats, split, poll, finds, lot, republicans, say, doing, responsibility, military, escalation, support, war, wars, voters, israels)",2


In [ ]:
display(articles_raw['week_start'].value_counts())

,count
week_start,
2025-11-18,18384
2025-02-04,218
2023-10-10,215
2023-12-05,193
2024-10-15,190
...,...
2020-06-02,1
2025-11-04,1
2025-06-17,1


In [ ]:
all_articles_en = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/MAP6114 - Machine Learning/MAP 6114 Project/Model_Cheyanne/Uploads to Batch Processing/all_articles_en.csv")

In [ ]:
all_articles_en.columns

# Removes duplicates of the articles within the same week (assuming they were updated)
all_articles_en_unique = all_articles_en.drop_duplicates(subset=['url', 'week_start', 'week_end'])

articles_raw_cleaned = articles_raw.drop_duplicates(subset='url', keep='first')
print(len(articles_raw))
print(len(articles_raw_cleaned))

articles_merged = articles_raw_cleaned.merge(all_articles_en_unique, on = 'url', how = 'left')

print(len(articles_merged) - len(articles_raw_cleaned))

duplicates = articles_merged['url'].value_counts()

print(len(duplicates[duplicates>1]))

40062
39451
70
69


There are 70 additional rows in the 'articles_merged' DF than in the 'articles_raw_cleaned' DF. This is because some articles were republished at later dates. For example, one article may be originally printed in February, but printed again in June. The assumption was made that the reprinted article continues to be relevant, thus both articles were kept within the dataset.

In [ ]:
display(articles_merged.columns)

Index(['title_x', 'date_x', 'keywords', 'url', 'country', 'week_start_x',
       'week_start_y', 'week_end', 'title_y', 'date_y', 'domain', 'language'],
      dtype='object')

In [ ]:
articles_merged.dtypes

,0
title_x,object
date_x,datetime64[ns]
keywords,object
url,object
country,object
week_start_x,datetime64[ns]
week_start_y,object
week_end,object
title_y,object
date_y,object


In [ ]:
articles_merged['language'].unique()

array(['English'], dtype=object)

In [ ]:
display(articles_merged)

,title_x,date_x,keywords,url,country,week_start_x,week_start_y,week_end,title_y,date_y,domain,language
0,National Defense Authorization Act – Implicati...,2025-11-19 00:00:00,"[provisions, authorization, uyghur, president,...",https://www.jdsupra.com/legalnews/national-def...,NaN,2025-11-18,2022-01-03,2022-01-09,National Defense Authorization Act – Implicati...,20220103T204500Z,jdsupra.com,English
1,Full diplomatic break? Decoding the latest Bid...,2021-12-30 05:00:00,"[impose, break, ushakov, bidenputin, latest, s...",https://www.washingtonexaminer.com/restoring-a...,NaN,2021-12-28,2022-01-03,2022-01-09,Full diplomatic break ? Decoding the latest Bi...,20220103T131500Z,washingtonexaminer.com,English
2,Canadian Bank Settles OFAC Charges For Potenti...,2025-11-19 00:00:00,"[settles, sanctions, apparent, kingpin, violat...",https://www.mondaq.com/unitedstates/export-con...,NaN,2025-11-18,2022-01-03,2022-01-09,Canadian Bank Settles OFAC Charges For Potenti...,20220103T113000Z,mondaq.com,English
3,RAB wants govt to resolve US sanctions,2025-11-19 00:00:00,"[resolve, authorities, sanctions, officials, r...",https://en.prothomalo.com/bangladesh/rab-wants...,NaN,2025-11-18,2022-01-03,2022-01-09,RAB wants govt to resolve US sanctions,20220103T123000Z,en.prothomalo.com,English
4,"Travel News: Latest News, Breaking Stories, an...",2022-01-03 00:00:00,"[latest, unless, india, breaking, rate, losses...",https://skift.com/2022/01/03/airbnb-settles-cu...,Cuba,2021-12-28,2022-01-03,2022-01-09,Airbnb Settles Cuba Sanctions Issue With U . S...,20220103T230000Z,skift.com,English
...,...,...,...,...,...,...,...,...,...,...,...,...
39516,Rise up for Gaza rally held in Huntsville,2025-10-05 00:00:00,"[gaza, organizer, israel, waff, world, hamas, ...",https://www.waff.com/2025/10/05/rise-up-gaza-r...,NaN,2025-09-30,2025-09-29,2025-10-05,Rise up for Gaza rally held in Huntsville,20251006T000000Z,waff.com,English
39517,"In search of FOI that barks and bites, by Taiw...",2025-10-05 11:23:54,"[foi, adisa, law, implementation, barks, taiwo...",https://theeagleonline.com.ng/in-search-of-foi...,NaN,2025-09-30,2025-09-29,2025-10-05,"In search of FOI that barks and bites , by Tai...",20251005T144500Z,theeagleonline.com.ng,English
39518,"Stop the Malicious Lies, Onanuga Tackles US Se...",2025-10-05 00:00:00,"[lies, christians, country, mass, senator, thi...",https://www.thisdaylive.com/2025/10/05/stop-th...,NaN,2025-09-30,2025-09-29,2025-10-05,"Stop the Malicious Lies , Onanuga Tackles US S...",20251005T081500Z,thisdaylive.com,English
39519,Ukraine war briefing: China providing Russia w...,2025-10-05 00:00:00,"[ukrainian, providing, claims, intelligence, m...",https://www.theguardian.com/world/2025/oct/05/...,NaN,2025-09-30,2025-09-29,2025-10-05,Ukraine war briefing : China providing Russia ...,20251005T020000Z,theguardian.com,English


In [ ]:
articles = articles_merged.drop(columns = ['title_x', 'date_x', 'week_start_x', 'week_end','date_y', 'domain', 'language'])
articles = articles.rename(columns = {'title_y': 'title', 'week_start_y': 'week_start'})

In [ ]:
display(articles)

,keywords,url,country,week_start,title
0,"[provisions, authorization, uyghur, president,...",https://www.jdsupra.com/legalnews/national-def...,NaN,2022-01-03,National Defense Authorization Act – Implicati...
1,"[impose, break, ushakov, bidenputin, latest, s...",https://www.washingtonexaminer.com/restoring-a...,NaN,2022-01-03,Full diplomatic break ? Decoding the latest Bi...
2,"[settles, sanctions, apparent, kingpin, violat...",https://www.mondaq.com/unitedstates/export-con...,NaN,2022-01-03,Canadian Bank Settles OFAC Charges For Potenti...
3,"[resolve, authorities, sanctions, officials, r...",https://en.prothomalo.com/bangladesh/rab-wants...,NaN,2022-01-03,RAB wants govt to resolve US sanctions
4,"[latest, unless, india, breaking, rate, losses...",https://skift.com/2022/01/03/airbnb-settles-cu...,Cuba,2022-01-03,Airbnb Settles Cuba Sanctions Issue With U . S...
...,...,...,...,...,...
39516,"[gaza, organizer, israel, waff, world, hamas, ...",https://www.waff.com/2025/10/05/rise-up-gaza-r...,NaN,2025-09-29,Rise up for Gaza rally held in Huntsville
39517,"[foi, adisa, law, implementation, barks, taiwo...",https://theeagleonline.com.ng/in-search-of-foi...,NaN,2025-09-29,"In search of FOI that barks and bites , by Tai..."
39518,"[lies, christians, country, mass, senator, thi...",https://www.thisdaylive.com/2025/10/05/stop-th...,NaN,2025-09-29,"Stop the Malicious Lies , Onanuga Tackles US S..."
39519,"[ukrainian, providing, claims, intelligence, m...",https://www.theguardian.com/world/2025/oct/05/...,NaN,2025-09-29,Ukraine war briefing : China providing Russia ...


In [ ]:
articles['week_start'].unique()

array(['2022-01-03', '2022-01-10', '2022-01-17', '2022-01-24',
       '2022-01-31', '2022-02-07', '2022-02-14', '2022-02-21',
       '2022-02-28', '2023-03-20', '2022-03-07', '2023-07-17',
       '2022-03-14', '2022-03-21', '2022-03-28', '2022-04-04',
       '2024-12-23', '2022-04-11', '2022-04-18', '2022-04-25',
       '2022-05-02', '2022-05-09', '2022-05-16', '2022-05-23',
       '2022-05-30', '2022-06-06', '2022-06-13', '2022-06-20',
       '2022-06-27', '2022-07-04', '2022-07-11', '2022-07-18',
       '2022-07-25', '2022-08-01', '2022-08-08', '2022-08-15',
       '2022-08-22', '2022-08-29', '2022-09-05', '2022-09-12',
       '2022-09-19', '2022-09-26', '2022-10-03', '2022-10-10',
       '2024-09-23', '2022-10-17', '2023-12-04', '2022-10-24',
       '2023-05-15', '2025-02-10', '2025-02-17', '2022-10-31',
       '2022-11-07', '2023-04-24', '2022-11-14', '2022-11-21',
       '2022-11-28', '2024-05-20', '2023-03-27', '2022-12-05',
       '2022-12-12', '2023-06-26', '2023-09-18', '2024-

In [ ]:
#  Set global export location - Used for all exports within the file unless explicitly overwritten.
destination_folder = '/content/drive/MyDrive/Colab Notebooks/MAP6114 - Machine Learning/MAP 6114 Project/Model_Cheyanne'

In [ ]:
def save_and_export_to_drive(df, source_file_name, destination_folder_name):
  # Save file
  df.to_csv(source_file_name, index = False)

  # Export file to drive
  shutil.copy(source_file_name, destination_folder_name)

  # Returns file names in the destination folder as a Python SList
  # Format: ['file1.csv file2.csv'] - Files are not delimited by commas
  files = !ls "{destination_folder}"

  # Extract the file names and split on the space to format the file names as a list.
  all_filenames = files[0].split()
  print(all_filenames)

  # Verify that the file was copied to the destination folder.
  if source_file_name in all_filenames:
    print(f"'{source_file_name}' successfully copied to '{destination_folder}'")
  else:
    print(f"'{source_file_name}' not found in '{destination_folder}'")

In [ ]:
save_and_export_to_drive(articles, 'articles.csv', destination_folder)

['all_articles_en_tagged.csv', 'MODEL_Keywords.ipynb']
'articles.csv' not found in '/content/drive/MyDrive/Colab Notebooks/MAP6114 - Machine Learning/MAP 6114 Project/Model_Cheyanne'
